# Initializer Dashboard -- Unified Analysis

One-stop-shop for evaluating initialization strategies. Configure everything at the top,
then run all cells to get:

1. **Geometry analysis**: PCA projections after multi-layer RP + ReLU
2. **Gradient flow analysis**: Mean row norms, zero proportions, dead neurons
3. **Summary statistics table**: Side-by-side comparison of all initializers

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import torch

from rp_study.config import ExperimentConfig, NetworkConfig, GradientExperimentConfig
from rp_study.data.loaders import get_data_loader
from rp_study.models.initializers import list_initializers
from rp_study.projections import multi_layer_rp_with_init
from rp_study.experiments.gradient_analysis import compare_initializations
from rp_study.visualization.gradient_plots import (
    compare_initializations_plot,
    plot_row_norm_per_layer,
)

print(f"Available initializers in registry: {list_initializers()}")

## Configuration

Modify these parameters to customize the experiments. To add a new initializer,
register it in `src/rp_study/models/initializers.py` and add it to `INIT_STRATEGIES` below.

In [ ]:
# ====== EXPERIMENT CONFIGURATION ======

SEED = 42
DATASET = "fashion_mnist"       # Options: "mnist", "fashion_mnist"
DATA_DIR = "../data"
NUM_SAMPLES = 2000              # Samples for gradient computation

# --- Initializers to compare ---
# These must match names in the registry (see list_initializers() above)
INIT_STRATEGIES = [
    "he",
    "row_centered_he",
    "row_centered_he_var_adj",
    "partial_centered_he",
    "orthogonal_he",
    "centered_with_dc_he",
    # "kernel_preserving",      # Uncomment (slow: ~200 optimizer steps per layer)
    # "row_centered_final",
    # "orthogonal_tuned",
]

# Display labels (for cleaner plot legends)
INIT_LABELS = {
    "he": "He",
    "row_centered_he": "Row-Centered",
    "row_centered_he_var_adj": "Row-Centered (Var Adj)",
    "partial_centered_he": "Partial Centered (a=0.5)",
    "orthogonal_he": "Orthogonal He",
    "orthogonal_tuned": "Orthogonal Tuned",
    "centered_with_dc_he": "Centered + DC",
    "kernel_preserving": "Kernel Preserving",
    "row_centered_final": "Row-Centered Final (1.65)",
    "uniform_he": "Uniform He",
}

# --- Geometry experiment ---
GEOM_LAYER_COUNTS = [1, 5, 10, 20]   # Number of RP+ReLU layers to apply
GEOM_WIDTH = None                      # None = same as input dim (square); or set an int

# --- Gradient experiment ---
GRAD_N_HIDDEN = 50                     # Number of hidden layers
GRAD_WIDTH = 784                       # Width of hidden layers
GRAD_OUTPUT_DIM = 1                    # Output dimension

# Derived: gradient experiment architecture
GRAD_LAYER_SIZES = [784] + [GRAD_WIDTH] * GRAD_N_HIDDEN + [GRAD_OUTPUT_DIM]

print(f"\nConfiguration:")
print(f"  Dataset: {DATASET}")
print(f"  Initializers: {len(INIT_STRATEGIES)}")
print(f"  Geometry layers: {GEOM_LAYER_COUNTS}")
print(f"  Gradient architecture: {GRAD_LAYER_SIZES[0]} -> [{GRAD_WIDTH}] x {GRAD_N_HIDDEN} -> {GRAD_OUTPUT_DIM}")

In [ ]:
# Load data
config = ExperimentConfig(seed=SEED, data_dir=DATA_DIR)
config.setup_seeds()

X, y = get_data_loader(
    dataset_name=DATASET,
    data_dir=DATA_DIR,
    train=True,
    flatten=True,
    as_numpy=True
)

print(f"Loaded {DATASET}: {X.shape}, {len(np.unique(y))} classes")

## 1. Geometry Analysis

Pass data through multi-layer RP + ReLU, then PCA to 2D for visualization.
This reveals whether the initialization preserves class structure (geometry) or collapses it.

In [ ]:
# Run geometry experiments for each initializer and layer count
print("Running geometry experiments...")
geom_results = {}

for init in INIT_STRATEGIES:
    label = INIT_LABELS.get(init, init)
    geom_results[init] = {}
    for n_layers in GEOM_LAYER_COUNTS:
        X_proj = multi_layer_rp_with_init(
            X, n_layers,
            init_strategy=init,
            width=GEOM_WIDTH,
            seed=SEED,
        )
        pca = PCA(n_components=2)
        geom_results[init][n_layers] = pca.fit_transform(X_proj)
    print(f"  {label}: done")

print("Done!")

In [ ]:
# Plot geometry grid: rows = initializers, columns = layer counts
n_rows = len(INIT_STRATEGIES)
n_cols = len(GEOM_LAYER_COUNTS)

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(4 * n_cols, 3.5 * n_rows))

# Handle single-row case
if n_rows == 1:
    axes = axes[np.newaxis, :]

scatter_kw = dict(s=3, c=y, cmap='viridis', alpha=0.7)

for i, init in enumerate(INIT_STRATEGIES):
    label = INIT_LABELS.get(init, init)
    for j, n_layers in enumerate(GEOM_LAYER_COUNTS):
        ax = axes[i, j]
        X_vis = geom_results[init][n_layers]
        ax.scatter(X_vis[:, 0], X_vis[:, 1], **scatter_kw)

        if i == 0:
            ax.set_title(f"{n_layers} Layers", fontsize=12)
        if j == 0:
            ax.set_ylabel(label, fontsize=10)

        ax.set_xticks([])
        ax.set_yticks([])
        ax.axis('equal')

plt.suptitle(
    f"{DATASET.upper()}: Effect of Initialization on Multi-Layer Square RP + ReLU Geometry",
    fontsize=14, y=1.01
)
plt.tight_layout()
plt.show()

## 2. Gradient Flow Analysis

Build a deep network with each initialization, perform one forward + backward pass,
and measure per-layer gradient statistics.

In [ ]:
# Run gradient analysis for all initializers
print(f"Running gradient analysis ({GRAD_N_HIDDEN} hidden layers, width={GRAD_WIDTH})...")
print(f"Architecture: {GRAD_LAYER_SIZES[:3]}...{GRAD_LAYER_SIZES[-3:]}")

grad_results = compare_initializations(
    layer_sizes=GRAD_LAYER_SIZES,
    init_strategies=INIT_STRATEGIES,
    num_samples=NUM_SAMPLES,
    dataset=DATASET,
    seed=SEED,
)

print("Done!")

In [ ]:
# Relabel results for cleaner plot legends
grad_results_labeled = {
    INIT_LABELS.get(k, k): v for k, v in grad_results.items()
}

# Mean gradient row norm per layer (log scale)
fig = compare_initializations_plot(
    grad_results_labeled,
    metric="mean_row_norm",
    exclude_output_layer=True,
    use_log_scale=True,
)
plt.title("Mean Gradient Row Norm per Layer (log scale, hidden layers only)")
plt.show()

In [ ]:
# Zero row proportion per layer (dead neurons in gradient space)
fig = compare_initializations_plot(
    grad_results_labeled,
    metric="zero_row_proportion",
    exclude_output_layer=True,
)
plt.title("Zero Gradient Row Proportion per Layer (hidden layers only)")
plt.show()

In [ ]:
# Zero gradient entry proportion
fig = compare_initializations_plot(
    grad_results_labeled,
    metric="zero_proportion",
    exclude_output_layer=True,
)
plt.title("Zero Gradient Entry Proportion per Layer (hidden layers only)")
plt.show()

## 3. Summary Statistics

Side-by-side comparison table of key metrics for each initialization strategy.

In [ ]:
# Print detailed comparison table
print("=" * 70)
print("COMPARISON RESULTS")
print("=" * 70)

for strategy, result in grad_results.items():
    label = INIT_LABELS.get(strategy, strategy)

    # Gradient entry zeros
    grad_zero_props = result.get_zero_gradient_proportions()
    avg_grad_zero = np.mean(list(grad_zero_props.values()))

    # Activation zeros (should be ~50% due to ReLU)
    act_zero_props = result.get_activation_zero_proportions()
    avg_act_zero = np.mean(list(act_zero_props.values()))

    # Truly inactive neurons (dead for ALL samples)
    inactive_props = result.get_truly_inactive_proportions()
    avg_inactive = np.mean(list(inactive_props.values()))

    # Gradient row norms
    row_norms = result.get_mean_row_norms()
    norm_vals = list(row_norms.values())
    # Exclude output layer for median
    hidden_norms = norm_vals[:-1] if len(norm_vals) > 1 else norm_vals
    mid = len(hidden_norms) // 2
    mid_norm = hidden_norms[mid] if hidden_norms else 0.0

    # Approximate per-layer gain (ratio of consecutive norms, early layers)
    if len(hidden_norms) >= 4:
        gains = [hidden_norms[i] / hidden_norms[i+1]
                 for i in range(min(10, len(hidden_norms)-1))
                 if hidden_norms[i+1] > 0]
        approx_gain = np.median(gains) if gains else float('nan')
    else:
        approx_gain = float('nan')

    print(f"\n{label.upper()}:")
    print(f"  Gradient entry zeros:   {avg_grad_zero:6.1%}")
    print(f"  Activation zeros:       {avg_act_zero:6.1%}  (expected ~50%)")
    print(f"  Truly inactive neurons: {avg_inactive:6.1%}  (dead neurons)")
    print(f"  Mid-network row norm:   {mid_norm:.2e}")
    print(f"  Approx per-layer gain:  {approx_gain:.3f}")

print("\n" + "=" * 70)

## 4. Per-Initializer Gradient Detail

Individual gradient row norm plots for each initializer (useful for detailed inspection).

In [ ]:
for strategy, result in grad_results.items():
    label = INIT_LABELS.get(strategy, strategy)
    fig = plot_row_norm_per_layer(
        result,
        exclude_output_layer=True,
        use_log_scale=True,
        figsize=(12, 4),
    )
    plt.title(f"Gradient Row Norm per Layer -- {label}")
    plt.show()